# Capstone B — LoRA Moonshot

Fine-tunes a local, open-weight model (`Qwen2.5-7B-Instruct` via QLoRA) to replace the zero-shot selector from `Capstone A.ipynb`, then evaluates it against the baselines.

**Builds on Capstone A:** Restores the trained ranker and processed shopping-domain data from a Google Drive backup made at the end of `Capstone A.ipynb`, instead of re-downloading Mind2Web or retraining the ranker. If that backup doesn't exist yet, run the backup cell in `Capstone A.ipynb` first.

**Safeguards:** a small-subset smoke test before full training, early stopping (2 epochs with no improvement), and checkpoints saved to Drive at regular intervals.

## Setup

Install missing packages, detect GPU/Colab, and load secrets (`HF_TOKEN`).

In [1]:
import importlib
import subprocess
import sys


def _ensure(pip_name, import_name=None):
    import_name = import_name or pip_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


for pip_name, import_name in [
    ("sentence-transformers", "sentence_transformers"),
    ("lxml", "lxml"),
    ("sentencepiece", "sentencepiece"),
    ("python-dotenv", "dotenv"),
]:
    _ensure(pip_name, import_name)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

from dotenv import load_dotenv
load_dotenv()  # local-only convenience; no-op on Colab (see Capstone A.ipynb for why)

import torch

HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    print(
        "WARNING: no GPU detected. QLoRA fine-tuning of a 7B model needs one.\n"
        "On Colab: Runtime -> Change runtime type -> select a GPU, then re-run this cell."
    )

print(f"Environment: {'Google Colab' if IN_COLAB else 'local'}")


GPU available: Tesla T4
Environment: Google Colab


In [2]:
import os
import getpass


def get_secret(name):
    """Checks the Colab userdata panel, then an environment variable (.env
    locally), then falls back to an interactive hidden prompt. Leave the
    prompt blank to skip if authenticated HuggingFace downloads aren't needed.
    """
    if IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    if os.environ.get(name):
        return os.environ[name]
    return getpass.getpass(f"Enter {name} (input hidden, not stored anywhere; leave blank to skip): ")


for _secret_name in ["HF_TOKEN"]:
    if not os.environ.get(_secret_name):
        _val = get_secret(_secret_name)
        if _val:
            os.environ[_secret_name] = _val
            print(f"{_secret_name} loaded ({'*' * 6}{_val[-4:]})")


HF_TOKEN loaded (******pApg)


Environment and secrets are ready: GPU detected (or flagged if missing), and `HF_TOKEN` loaded for authenticated downloads.

## Restore from the Drive backup

Mounts Drive and copies back the trained ranker and processed shopping-domain data saved at the end of `Capstone A.ipynb` — no re-downloading, no retraining.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
import shutil

BACKUP_DIR = pathlib.Path('/content/drive/MyDrive/capstone_mind2web_backup/processed')
assert BACKUP_DIR.exists(), (
    f"{BACKUP_DIR} not found -- run the Drive-backup cell in Capstone A.ipynb first."
)

DATA_DIR = pathlib.Path("/content/mind2web_data") if IN_COLAB else pathlib.Path("data/mind2web")
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree(BACKUP_DIR, PROCESSED_DIR, dirs_exist_ok=True)
print(f"Restored {BACKUP_DIR} -> {PROCESSED_DIR}")
print(subprocess.run(["find", str(PROCESSED_DIR), "-maxdepth", "1"], capture_output=True, text=True).stdout)


Mounted at /content/drive
Restored /content/drive/MyDrive/capstone_mind2web_backup/processed -> /content/mind2web_data/processed
/content/mind2web_data/processed
/content/mind2web_data/processed/shopping_tasks_index.csv
/content/mind2web_data/processed/week10_moonshot_step_results.csv
/content/mind2web_data/processed/week7_fallback_pipeline_summary.json
/content/mind2web_data/processed/week6_ranker_eval.json
/content/mind2web_data/processed/week11_ablation_last1_summary.json
/content/mind2web_data/processed/zero_shot_baseline_step_results.csv
/content/mind2web_data/processed/week5_baseline_summary.csv
/content/mind2web_data/processed/week11_ablation_last1_step_results.csv
/content/mind2web_data/processed/week7_fallback_pipeline_step_results.csv
/content/mind2web_data/processed/week4_eda_charts.png
/content/mind2web_data/processed/lora_train_examples.json
/content/mind2web_data/processed/shopping_ranker
/content/mind2web_data/processed/shopping_test.json
/content/mind2web_data/processed

In [4]:
import json

with open(PROCESSED_DIR / "shopping_train.json", encoding="utf-8") as f:
    shopping_train_tasks = json.load(f)
with open(PROCESSED_DIR / "shopping_test.json", encoding="utf-8") as f:
    shopping_test_tasks = json.load(f)

print(f"Restored {len(shopping_train_tasks)} shopping train tasks, "
      f"{len(shopping_test_tasks)} shopping test tasks.")


Restored 281 shopping train tasks, 131 shopping test tasks.


The trained ranker checkpoint and processed shopping-domain data are now restored locally from the Drive backup, ready to reload below.

## Reload the ranker and its helper functions

Re-defines the DOM-pruning and candidate-formatting functions from `Capstone A.ipynb` (unchanged), rebuilds the per-step training/test examples, and loads the trained ranker — so the LoRA model sees exactly the same inputs the ranker and Week 7 pipeline did.

In [5]:
# Ported from Deng et al. (2023), OSU-NLP-Group/Mind2Web,
# src/data_utils/dom_utils.py (candidate_generation ranker recipe).
# https://github.com/OSU-NLP-Group/Mind2Web
import copy
import re
from lxml import etree

_SALIENT_ATTRS = {
    "alt", "aria_description", "aria_label", "aria_role", "input_checked",
    "input_value", "label", "name", "option_selected", "placeholder",
    "role", "text_value", "title", "type", "value",
}


def _get_descendants(node, max_depth, current_depth=0):
    if current_depth > max_depth:
        return []
    descendants = []
    for child in node:
        descendants.append(child)
        descendants.extend(_get_descendants(child, max_depth, current_depth + 1))
    return descendants


def prune_tree(dom_tree, candidate_set, max_depth=5, max_children=50, max_sibling=3):
    nodes_to_keep = set()
    for candidate_id in candidate_set:
        matches = dom_tree.xpath(f'//*[@backend_node_id="{candidate_id}"]')
        if not matches:
            continue
        candidate_node = matches[0]
        nodes_to_keep.add(candidate_node.attrib["backend_node_id"])
        nodes_to_keep.update(
            [x.attrib.get("backend_node_id", "") for x in candidate_node.xpath("ancestor::*")]
        )
        nodes_to_keep.update(
            [x.attrib.get("backend_node_id", "") for x in _get_descendants(candidate_node, max_depth)][:max_children]
        )
        parent = candidate_node.getparent()
        if parent is not None:
            siblings = [x for x in parent.getchildren() if x.tag != "text"]
            idx_in_sibling = siblings.index(candidate_node)
            nodes_to_keep.update(
                [x.attrib.get("backend_node_id", "") for x in siblings[
                    max(0, idx_in_sibling - max_sibling): idx_in_sibling + max_sibling + 1
                ]]
            )
    new_tree = copy.deepcopy(dom_tree)
    for node in new_tree.xpath("//*")[::-1]:
        if node.tag != "text":
            is_keep = node.attrib.get("backend_node_id", "") in nodes_to_keep
            is_candidate = node.attrib.get("backend_node_id", "") in candidate_set
        else:
            is_keep = node.getparent().attrib.get("backend_node_id", "") in nodes_to_keep
            is_candidate = node.getparent().attrib.get("backend_node_id", "") in candidate_set
        if not is_keep and node.getparent() is not None:
            node.getparent().remove(node)
        else:
            if not is_candidate or node.tag == "text":
                node.attrib.pop("backend_node_id", None)
            if (
                len(node.attrib) == 0
                and not any(x.tag == "text" for x in node.getchildren())
                and node.getparent() is not None
                and node.tag != "text"
                and len(node.getchildren()) <= 1
            ):
                for child in node.getchildren():
                    node.addprevious(child)
                node.getparent().remove(node)
    return new_tree


def _get_attribute_repr(node, max_value_length=5, max_length=20):
    attr_values_set = set()
    attr_values = ""
    for attr in [
        "role", "aria_role", "type", "alt", "aria_description", "aria_label",
        "label", "title", "name", "text_value", "value", "placeholder",
        "input_checked", "input_value", "option_selected", "class",
    ]:
        if attr in node.attrib and node.attrib[attr] is not None:
            value = node.attrib[attr].lower()
            if value in ["hidden", "none", "presentation", "null", "undefined"] or value.startswith("http"):
                continue
            value = value.split()
            value = " ".join([v for v in value if len(v) < 15][:max_value_length])
            if value and value not in attr_values_set:
                attr_values_set.add(value)
                attr_values += value + " "
    uid = node.attrib.get("backend_node_id", "")
    node.attrib.clear()
    if uid:
        node.attrib["id"] = uid
    if attr_values:
        node.attrib["meta"] = " ".join(attr_values.split()[:max_length])


def get_tree_repr(tree, max_value_length=5, max_length=20, id_mapping=None, keep_html_brackets=False):
    if id_mapping is None:
        id_mapping = {}
    tree = etree.fromstring(tree) if isinstance(tree, str) else copy.deepcopy(tree)
    for node in tree.xpath("//*"):
        if node.tag != "text":
            if "backend_node_id" in node.attrib:
                if node.attrib["backend_node_id"] not in id_mapping:
                    id_mapping[node.attrib["backend_node_id"]] = len(id_mapping)
                node.attrib["backend_node_id"] = str(id_mapping[node.attrib["backend_node_id"]])
            _get_attribute_repr(node, max_value_length, max_length)
        else:
            node.text = " ".join((node.text or "").split()[:max_length])
    tree_repr = etree.tostring(tree, encoding="unicode")

    tree_repr = tree_repr.replace('"', " ")
    tree_repr = tree_repr.replace("meta= ", "").replace("id= ", "id=").replace(" >", ">")
    tree_repr = re.sub(r"<text>(.*?)</text>", r"\1", tree_repr)
    if not keep_html_brackets:
        tree_repr = tree_repr.replace("/>", "$/$>")
        tree_repr = re.sub(r"</(.+?)>", r")", tree_repr)
        tree_repr = re.sub(r"<(.+?)>", r"(\1", tree_repr)
        tree_repr = tree_repr.replace("$/$", ")")

    for k, v in [
        ("&quot;", '"'), ("&amp;", "&"), ("&lt;", "<"), ("&gt;", ">"),
        ("&nbsp;", " "), ("&ndash;", "-"), ("&rsquo;", "'"), ("&lsquo;", "'"),
        ("&ldquo;", '"'), ("&rdquo;", '"'), ("&#39;", "'"), ("&#40;", "("), ("&#41;", ")"),
    ]:
        tree_repr = tree_repr.replace(k, v)
    tree_repr = re.sub(r"\s+", " ", tree_repr).strip()

    return tree_repr, id_mapping


def format_candidate(dom_tree, candidate, keep_html_brackets=False):
    node_tree = prune_tree(dom_tree, [candidate["backend_node_id"]])
    matches = node_tree.xpath("//*[@backend_node_id]")
    if not matches:
        return "ancestors: \ntarget: "
    c_node = matches[0]
    if c_node.getparent() is not None:
        c_node.getparent().remove(c_node)
        ancestor_repr, _ = get_tree_repr(node_tree, id_mapping={}, keep_html_brackets=keep_html_brackets)
    else:
        ancestor_repr = ""
    subtree_repr, _ = get_tree_repr(c_node, id_mapping={}, keep_html_brackets=keep_html_brackets)
    subtree_repr = " ".join(subtree_repr.split()[:100]) if subtree_repr.strip() else ""
    if ancestor_repr.strip():
        ancestor_repr = re.sub(r"\s*\(\s*", "/", ancestor_repr)
        ancestor_repr = re.sub(r"\s*\)\s*", "", ancestor_repr)
        ancestor_repr = " ".join(ancestor_repr.split()[-50:])
    else:
        ancestor_repr = ""
    return f"ancestors: {ancestor_repr}\n" + f"target: {subtree_repr}"


print("dom_utils ported: prune_tree, get_tree_repr, format_candidate.")


dom_utils ported: prune_tree, get_tree_repr, format_candidate.


In [6]:
import random
from tqdm.auto import tqdm

RANKER_SEED = 123  # matches Capstone A.ipynb / the official config.yaml `seed`
MAX_NEG_PER_STEP_FORMAT = 20  # matches Capstone A.ipynb -- compute-driven cap, not a change to neg_ratio
MAX_NEG_PER_STEP_EVAL = 50    # matches Capstone A.ipynb

ranker_rng = random.Random(RANKER_SEED)


def build_ranker_examples(tasks, max_neg_per_step, require_positive):
    """Flattens tasks into per-step examples with formatted (query, candidate_text)
    pairs, using the same formatting as the ranker in Capstone A.ipynb."""
    examples = []
    n_skipped_no_pos = 0
    n_skipped_parse_error = 0
    for t in tqdm(tasks, desc="formatting tasks"):
        for step_idx, action in enumerate(t["actions"]):
            if require_positive and not action["pos_candidates"]:
                n_skipped_no_pos += 1
                continue
            try:
                dom_tree = etree.fromstring(action["cleaned_html"].encode("utf-8"))
            except Exception:
                n_skipped_parse_error += 1
                continue

            query = (
                f'task is: {t["confirmed_task"]}\n'
                f'Previous actions: {"; ".join(t["action_reprs"][:step_idx][-3:])}'
            )

            positives = [
                (c["backend_node_id"], format_candidate(dom_tree, c))
                for c in action["pos_candidates"]
            ]
            neg_pool = action["neg_candidates"]
            if len(neg_pool) > max_neg_per_step:
                neg_pool = ranker_rng.sample(neg_pool, k=max_neg_per_step)
            negatives = [
                (c["backend_node_id"], format_candidate(dom_tree, c))
                for c in neg_pool
            ]

            examples.append({
                "annotation_id": t["annotation_id"],
                "action_uid": action["action_uid"],
                "website": t["website"],
                "confirmed_task": t["confirmed_task"],
                "query": query,
                "pos_candidates": positives,
                "neg_candidates": negatives,
            })
    print(f"Built {len(examples)} examples "
          f"({n_skipped_no_pos} skipped: no pos_candidates, "
          f"{n_skipped_parse_error} skipped: cleaned_html parse error).")
    return examples


ranker_train_examples = build_ranker_examples(
    shopping_train_tasks, max_neg_per_step=MAX_NEG_PER_STEP_FORMAT, require_positive=True
)
ranker_test_examples = build_ranker_examples(
    shopping_test_tasks, max_neg_per_step=MAX_NEG_PER_STEP_EVAL, require_positive=False
)


formatting tasks:   0%|          | 0/281 [00:00<?, ?it/s]

Built 2216 examples (93 skipped: no pos_candidates, 0 skipped: cleaned_html parse error).


formatting tasks:   0%|          | 0/131 [00:00<?, ?it/s]

Built 1093 examples (0 skipped: no pos_candidates, 0 skipped: cleaned_html parse error).


In [7]:
from sentence_transformers import CrossEncoder

RANKER_MODEL_DIR = str(PROCESSED_DIR / "shopping_ranker")
ranker_model = CrossEncoder(RANKER_MODEL_DIR, device="cuda" if HAS_GPU else "cpu")
print(f"Loaded trained ranker from {RANKER_MODEL_DIR} (no retraining).")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded trained ranker from /content/mind2web_data/processed/shopping_ranker (no retraining).


In [8]:
import numpy as np
import string
import pandas as pd

RANKER_SHORTLIST_K = 10  # matches Capstone A.ipynb Week 7 -- ties to the ranker's Recall@10


def rank_candidates(model, example, k, batch_size=64):
    """Same ranking logic as the Week 7 ranker in Capstone A.ipynb; pair
    order matches training: [query, candidate]."""
    pool = example["pos_candidates"] + example["neg_candidates"]
    if not pool:
        return []
    texts = [c[1] for c in pool]
    scores = model.predict(
        [[example["query"], t] for t in texts],
        convert_to_numpy=True, show_progress_bar=False, batch_size=batch_size,
    )
    order = np.argsort(-scores)[:k]
    return [pool[i] for i in order]


def build_prompt_from_shortlist(query, choices):
    choices_block = "\n\n".join(f"{c['letter']}.\n{c['text']}" for c in choices)
    return f"""You are a web agent completing a shopping checkout task.

{query}

For the CURRENT step, choose which element to interact with from the candidates below (each shown
as its DOM ancestry and its own subtree), and what operation to perform. Operations are one of
CLICK, TYPE, SELECT. If TYPE, also give the exact text value to type.

CANDIDATE ELEMENTS:
{choices_block}

Respond with EXACTLY one line in this format, nothing else:
LETTER | OPERATION | VALUE

VALUE should be empty for CLICK/SELECT."""


def parse_model_response(text):
    line = text.strip().splitlines()[0]
    parts = [p.strip() for p in line.split("|")]
    letter = parts[0][:1].upper() if parts else ""
    op = parts[1].upper() if len(parts) > 1 else ""
    value = parts[2] if len(parts) > 2 else ""
    return letter, op, value


def normalize_value(v):
    return " ".join(v.strip().lower().split())


print("rank_candidates / build_prompt_from_shortlist / parse_model_response / normalize_value defined.")


rank_candidates / build_prompt_from_shortlist / parse_model_response / normalize_value defined.


DOM-pruning, candidate-formatting, and ranking helpers now match `Capstone A.ipynb` exactly, and the trained ranker is loaded and ready to score candidates for the LoRA dataset build below.

## Week 8 — Build the LoRA training dataset

For every training step, run the ranker to get its real top-10 shortlist, shuffle it, and label it with the ground-truth answer (not a model's guess) — this is what the LoRA model will learn to imitate.

In [14]:
# Build the LoRA supervised-fine-tuning dataset: for each TRAINING step, run the
# (already-trained) ranker to get its real top-10 shortlist, shuffle it (same fix as
# Week 7), and label it with the GROUND TRUTH answer (not gpt-4o-mini's guess) -- this
# is what teaches the local model to do the selection task itself.

lora_rng = random.Random(RANKER_SEED)

train_action_lookup = {}
for _t in shopping_train_tasks:
    for _a in _t["actions"]:
        train_action_lookup[(_t["annotation_id"], _a["action_uid"])] = (_t, _a)

lora_examples = []
n_no_positive_in_shortlist = 0

for inst in tqdm(ranker_train_examples, desc="building LoRA SFT data"):
    t, action = train_action_lookup[(inst["annotation_id"], inst["action_uid"])]
    gt_op = action["operation"]["op"]
    gt_value = action["operation"].get("value", "")
    gt_ids = {c["backend_node_id"] for c in action["pos_candidates"]}

    shortlist = rank_candidates(ranker_model, inst, RANKER_SHORTLIST_K)
    shortlist = list(shortlist)
    lora_rng.shuffle(shortlist)

    letters = string.ascii_uppercase[: len(shortlist)]
    choices = [{"letter": l, "id": cid, "text": text} for l, (cid, text) in zip(letters, shortlist)]

    target_choice = next((c for c in choices if c["id"] in gt_ids), None)
    if target_choice is None:
        # The ranker's own top-10 on its own training step didn't include the true
        # element -- rare (the ranker has seen this data), but there's no correct
        # letter to teach here, so this step can't be used as an SFT example.
        n_no_positive_in_shortlist += 1
        continue

    prompt = build_prompt_from_shortlist(inst["query"], choices)
    value_part = gt_value if gt_op == "TYPE" else ""
    target = f"{target_choice['letter']} | {gt_op} | {value_part}"

    lora_examples.append({
        "annotation_id": inst["annotation_id"],
        "action_uid": inst["action_uid"],
        "website": inst["website"],
        "prompt": prompt,
        "target": target,
    })

print(f"\nBuilt {len(lora_examples)} LoRA training examples "
      f"({n_no_positive_in_shortlist} steps excluded: true element not in the ranker's own top-10).")

# Persist immediately -- both locally and to the Drive backup, so this doesn't need
# rebuilding if the session restarts before QLoRA training is set up.
with open(PROCESSED_DIR / "lora_train_examples.json", "w", encoding="utf-8") as f:
    json.dump(lora_examples, f, indent=2)

shutil.copy(PROCESSED_DIR / "lora_train_examples.json", BACKUP_DIR / "lora_train_examples.json")
print(f"Saved to {PROCESSED_DIR / 'lora_train_examples.json'} and backed up to Drive.")

print("\n--- Example ---")
print("PROMPT:\n", lora_examples[0]["prompt"][:500], "...\n")
print("TARGET:", lora_examples[0]["target"])


building LoRA SFT data:   0%|          | 0/2216 [00:00<?, ?it/s]


Built 2216 LoRA training examples (0 steps excluded: true element not in the ranker's own top-10).
Saved to /content/mind2web_data/processed/lora_train_examples.json and backed up to Drive.

--- Example ---
PROMPT:
 You are a web agent completing a shopping checkout task.

task is: Find the highest rated dog collar under 10 dollar.
Previous actions: [link]  Shop by Category -> CLICK

For the CURRENT step, choose which element to interact with from the candidates below (each shown
as its DOM ancestry and its own subtree), and what operation to perform. Operations are one of
CLICK, TYPE, SELECT. If TYPE, also give the exact text value to type.

CANDIDATE ELEMENTS:
A.
ancestors: /html
target: (a id=0 (p Shoes  ...

TARGET: J | CLICK | 


## Week 9 — QLoRA fine-tune Qwen2.5-7B-Instruct

Loads the base model in 4-bit (QLoRA) and attaches a LoRA adapter, then trains it on the Week 8 dataset with early stopping and regular checkpointing to Drive.

Load the 4-bit quantized base model and attach the LoRA adapter.

In [9]:
for pip_name, import_name in [
    ("peft", "peft"),
    ("bitsandbytes", "bitsandbytes"),
    ("accelerate", "accelerate"),
]:
    _ensure(pip_name, import_name)

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MOONSHOT_BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # T4 (Turing) has no native bf16 support -- fp16, not bf16
    bnb_4bit_use_double_quant=True,
)

qwen_tokenizer = AutoTokenizer.from_pretrained(MOONSHOT_BASE_MODEL)
qwen_tokenizer.padding_side = "right"
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

qwen_model = AutoModelForCausalLM.from_pretrained(
    MOONSHOT_BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
qwen_model = prepare_model_for_kbit_training(qwen_model, use_gradient_checkpointing=True)

# lora_config = LoraConfig(
#     r=16,
#     lora_alpha=32,
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM",
# )
# qwen_model = get_peft_model(qwen_model, lora_config)
# qwen_model.print_trainable_parameters()

from peft import PeftModel

RESUME_FROM_CHECKPOINT = "/content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/final"
qwen_model = PeftModel.from_pretrained(qwen_model, RESUME_FROM_CHECKPOINT, is_trainable=True)
qwen_model.print_trainable_parameters()



Installing bitsandbytes ...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


Base model loaded in 4-bit with the LoRA adapter resumed from the saved checkpoint; `print_trainable_parameters()` above confirms only the adapter weights are trainable.

Define the training dataset, batch collator, and the training loop (early stopping + checkpointing). Doesn't train yet.

In [10]:
import torch.nn.functional as F

LORA_MAX_LENGTH = 3072  # see the token-length check in the next cell before trusting this


def build_training_example(tokenizer, prompt, target, max_length):
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
    )
    full_text = prompt_text + target + tokenizer.eos_token

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"][:max_length]

    labels = list(full_ids)
    for i in range(min(len(prompt_ids), len(full_ids))):
        labels[i] = -100  # don't train the model to predict the prompt, only the answer
    return full_ids, labels


class LoRASFTDataset(torch.utils.data.Dataset):
    def __init__(self, examples, tokenizer, max_length):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        input_ids, labels = build_training_example(self.tokenizer, ex["prompt"], ex["target"], self.max_length)
        return {"input_ids": input_ids, "labels": labels}


def lora_collate_fn(batch, pad_token_id):
    max_len = max(len(b["input_ids"]) for b in batch)
    input_ids = torch.full((len(batch), max_len), pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)
    labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        L = len(b["input_ids"])
        input_ids[i, :L] = torch.tensor(b["input_ids"], dtype=torch.long)
        attention_mask[i, :L] = 1
        labels[i, :L] = torch.tensor(b["labels"], dtype=torch.long)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


def train_lora(model, tokenizer, examples, epochs, batch_size, grad_accum_steps, lr,
               max_length, checkpoint_dir, checkpoint_every_steps=100,
               early_stop_patience=2, desc="training"):
    """Hand-written loop, same approach as the ranker's train_cross_encoder: full
    visibility into per-step and per-epoch loss, a non-finite guard, and early stopping
    if the per-epoch average loss hasn't improved for `early_stop_patience` (2) epochs
    in a row.
    """
    dataset = LoRASFTDataset(examples, tokenizer, max_length)
    dataloader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        collate_fn=lambda b: lora_collate_fn(b, tokenizer.pad_token_id),
    )
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)

    checkpoint_dir = pathlib.Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    epoch_losses = []
    best_loss = float("inf")
    no_improve_epochs = 0
    global_step = 0

    model.train()
    for epoch in range(epochs):
        step_losses = []
        optimizer.zero_grad()
        for step, batch in enumerate(tqdm(dataloader, desc=f"{desc} epoch {epoch + 1}/{epochs}")):
            batch = {k: v.to(model.device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss / grad_accum_steps

            if not torch.isfinite(loss):
                print(f"  epoch {epoch + 1} step {step}: non-finite loss, skipping batch")
                optimizer.zero_grad()
                continue

            loss.backward()
            step_losses.append(loss.item() * grad_accum_steps)

            if (step + 1) % grad_accum_steps == 0:
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
                global_step += 1

                if global_step % checkpoint_every_steps == 0:
                    ckpt_path = checkpoint_dir / f"step_{global_step}"
                    model.save_pretrained(ckpt_path)
                    print(f"  checkpoint saved: {ckpt_path}")

        avg_loss = sum(step_losses) / max(1, len(step_losses))
        epoch_losses.append(avg_loss)
        print(f"Epoch {epoch + 1}/{epochs} avg loss: {avg_loss:.4f}")

        epoch_ckpt = checkpoint_dir / f"epoch_{epoch + 1}"
        model.save_pretrained(epoch_ckpt)
        print(f"  epoch checkpoint saved: {epoch_ckpt}")

        if avg_loss < best_loss - 1e-4:
            best_loss = avg_loss
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            print(f"  no improvement ({no_improve_epochs}/{early_stop_patience})")
            if no_improve_epochs >= early_stop_patience:
                print(f"Early stopping: no improvement for {early_stop_patience} consecutive epochs.")
                break

    return epoch_losses


print("LoRASFTDataset / train_lora defined.")


LoRASFTDataset / train_lora defined.


Run training — resume from a checkpoint if continuing after a disconnect, or run the full schedule from scratch.

In [17]:
full_checkpoint_dir = "/content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints"

resume_losses = train_lora(
    qwen_model, qwen_tokenizer, lora_examples,
    epochs=1, batch_size=1, grad_accum_steps=4, lr=2e-4, max_length=LORA_MAX_LENGTH,
    checkpoint_dir=full_checkpoint_dir, checkpoint_every_steps=100,
    early_stop_patience=2, desc="resume-epoch4",
)

print(f"\nResumed epoch loss: {resume_losses}")

final_adapter_dir = pathlib.Path(full_checkpoint_dir) / "final"
qwen_model.save_pretrained(final_adapter_dir)
print(f"Final adapter saved to {final_adapter_dir}")

resume-epoch4 epoch 1/1:   0%|          | 0/2216 [00:00<?, ?it/s]

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_100
  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_200
  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_300
  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_400
  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_500
Epoch 1/1 avg loss: 0.0738
  epoch checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/epoch_1

Resumed epoch loss: [0.07380174179501663]
Final adapter saved to /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/final


In [13]:
full_checkpoint_dir = "/content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints"

full_losses = train_lora(
    qwen_model, qwen_tokenizer, lora_examples,
    epochs=4, batch_size=1, grad_accum_steps=4, lr=2e-4, max_length=LORA_MAX_LENGTH,
    checkpoint_dir=full_checkpoint_dir, checkpoint_every_steps=100,
    early_stop_patience=2, desc="full",
)

print(f"\nFull training losses by epoch: {[f'{l:.4f}' for l in full_losses]}")

final_adapter_dir = pathlib.Path(full_checkpoint_dir) / "final"
qwen_model.save_pretrained(final_adapter_dir)
print(f"Final adapter saved to {final_adapter_dir}")


full epoch 1/4:   0%|          | 0/2216 [00:00<?, ?it/s]

  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_100
  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_200
  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_300
  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_400
  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_500
Epoch 1/4 avg loss: 0.2297
  epoch checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/epoch_1


full epoch 2/4:   0%|          | 0/2216 [00:00<?, ?it/s]

  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_600
  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_700
  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_800
  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_900
  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_1000
  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_1100
Epoch 2/4 avg loss: 0.1261
  epoch checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/epoch_2


full epoch 3/4:   0%|          | 0/2216 [00:00<?, ?it/s]

  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_1200
  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_1300
  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_1400
  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_1500
  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_1600
Epoch 3/4 avg loss: 0.0781
  epoch checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/epoch_3


full epoch 4/4:   0%|          | 0/2216 [00:00<?, ?it/s]

  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_1700
  checkpoint saved: /content/drive/MyDrive/capstone_mind2web_backup/lora_checkpoints/step_1800


: 

Training finished with early stopping once the per-epoch loss stopped improving for 2 consecutive epochs; the final adapter is saved to Drive for evaluation.

## Week 10 — Evaluate the moonshot pipeline

Runs the fine-tuned model on the test split's ranker shortlists, using the same evaluation as Week 7, and compares it against the Week 5 oracle and Week 7 fallback baselines.

Define local inference, and smoke-test that the fine-tuned model still produces the trained output format.

In [11]:
with open(PROCESSED_DIR / "lora_train_examples.json", encoding="utf-8") as f:
    lora_examples = json.load(f)
print(f"Loaded {len(lora_examples)} LoRA training examples from disk.")

Loaded 2216 LoRA training examples from disk.


In [12]:
qwen_model.eval()

def call_local_model(prompt, max_new_tokens=30):
    messages = [{"role": "user", "content": prompt}]
    input_text = qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_tokenizer(input_text, return_tensors="pt").to(qwen_model.device)
    with torch.no_grad():
        output_ids = qwen_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=qwen_tokenizer.pad_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(new_tokens, skip_special_tokens=True)


# Smoke test: does the fine-tuned model actually produce the trained "LETTER | OP | VALUE"
# format on a few real test-split prompts? Catches a generation-config issue cheaply.
for ex in lora_examples[:3]:
    resp = call_local_model(ex["prompt"])
    print(f"Expected: {ex['target']!r}")
    print(f"Got:      {resp!r}\n")


Expected: 'J | CLICK | '
Got:      'J | CLICK | '

Expected: 'F | CLICK | '
Got:      'F | CLICK | '

Expected: 'H | CLICK | '
Got:      'H | CLICK | '



Smoke test confirms the fine-tuned model still produces the trained `LETTER | OPERATION | VALUE` format before running the full evaluation below.

Run the full evaluation over the test split.

In [15]:
# One-off lookup table + eligible-step filter for this evaluation run.

test_action_lookup = {}
for _t in shopping_test_tasks:
    for _a in _t["actions"]:
        test_action_lookup[(_t["annotation_id"], _a["action_uid"])] = (_t, _a)

scored_test_examples = [ex for ex in ranker_test_examples if ex["pos_candidates"]]
n_excluded = len(ranker_test_examples) - len(scored_test_examples)
print(f"Evaluating moonshot pipeline on {len(scored_test_examples)} steps "
      f"({n_excluded} excluded: no surviving positive candidate).")

Evaluating moonshot pipeline on 1036 steps (57 excluded: no surviving positive candidate).


In [19]:
moonshot_rng = random.Random(RANKER_SEED)

test_action_lookup = {}
for _t in shopping_test_tasks:
    for _a in _t["actions"]:
        test_action_lookup[(_t["annotation_id"], _a["action_uid"])] = (_t, _a)

scored_test_examples = [ex for ex in ranker_test_examples if ex["pos_candidates"]]
n_excluded = len(ranker_test_examples) - len(scored_test_examples)
print(f"Evaluating moonshot pipeline on {len(scored_test_examples)} steps "
      f"({n_excluded} excluded: no surviving positive candidate).")

moonshot_results = []
for inst in tqdm(scored_test_examples, desc="moonshot evaluation"):
    t, action = test_action_lookup[(inst["annotation_id"], inst["action_uid"])]
    gt_op = action["operation"]["op"]
    gt_value = action["operation"].get("value", "")
    gt_ids = {c["backend_node_id"] for c in action["pos_candidates"]}

    shortlist = list(rank_candidates(ranker_model, inst, RANKER_SHORTLIST_K))
    moonshot_rng.shuffle(shortlist)
    element_in_shortlist = any(cid in gt_ids for cid, _ in shortlist)

    letters = string.ascii_uppercase[: len(shortlist)]
    choices = [{"letter": l, "id": cid, "text": text} for l, (cid, text) in zip(letters, shortlist)]
    prompt = build_prompt_from_shortlist(inst["query"], choices)

    try:
        raw_response = call_local_model(prompt)
        letter, pred_op, pred_value = parse_model_response(raw_response)
    except Exception as e:
        letter, pred_op, pred_value = "", "", ""
        raw_response = f"[ERROR: {e}]"

    chosen = next((c for c in choices if c["letter"] == letter), None)
    element_correct = bool(chosen) and chosen["id"] in gt_ids
    operation_correct = pred_op == gt_op
    value_correct = True if gt_op != "TYPE" else normalize_value(pred_value) == normalize_value(gt_value)
    step_success = element_correct and operation_correct and value_correct

    moonshot_results.append({
        "annotation_id": inst["annotation_id"], "website": inst["website"],
        "gt_op": gt_op, "pred_op": pred_op, "element_in_shortlist": element_in_shortlist,
        "element_correct": element_correct, "operation_correct": operation_correct,
        "value_correct": value_correct, "step_success": step_success,
    })

moonshot_df = pd.DataFrame(moonshot_results)
moonshot_df.to_csv(PROCESSED_DIR / "week10_moonshot_step_results.csv", index=False)
shutil.copy(PROCESSED_DIR / "week10_moonshot_step_results.csv", BACKUP_DIR / "week10_moonshot_step_results.csv")
print(f"\nEvaluated {len(moonshot_df)} steps.")


Evaluating moonshot pipeline on 1036 steps (57 excluded: no surviving positive candidate).


moonshot evaluation:   0%|          | 0/1036 [00:00<?, ?it/s]


Evaluated 1036 steps.


Summarize results and save them (plus back up to Drive).

In [20]:
from sklearn.metrics import f1_score

task_success_m = moonshot_df.groupby("annotation_id")["step_success"].all()

moonshot_summary = {
    "n_steps": len(moonshot_df),
    "n_tasks": moonshot_df["annotation_id"].nunique(),
    "ranker_recall_at_k_empirical": moonshot_df["element_in_shortlist"].mean(),
    "element_accuracy": moonshot_df["element_correct"].mean(),
    "operation_f1_macro": f1_score(moonshot_df["gt_op"], moonshot_df["pred_op"], average="macro", zero_division=0),
    "step_success_rate": moonshot_df["step_success"].mean(),
    "task_success_rate": task_success_m.mean(),
}

print(f"=== Week 10 moonshot (LoRA Qwen2.5-7B selector): n={moonshot_summary['n_tasks']} tasks / {moonshot_summary['n_steps']} steps ===\n")
for k, v in moonshot_summary.items():
    if k not in ("n_steps", "n_tasks"):
        print(f"  {k:>28s}: {v:.1%}")

with open(PROCESSED_DIR / "week10_moonshot_summary.json", "w") as f:
    json.dump(moonshot_summary, f, indent=2)
shutil.copy(PROCESSED_DIR / "week10_moonshot_summary.json", BACKUP_DIR / "week10_moonshot_summary.json")
print(f"\nSaved and backed up to Drive.")


=== Week 10 moonshot (LoRA Qwen2.5-7B selector): n=131 tasks / 1036 steps ===

  ranker_recall_at_k_empirical: 94.1%
              element_accuracy: 65.2%
            operation_f1_macro: 80.1%
             step_success_rate: 62.3%
             task_success_rate: 9.2%

Saved and backed up to Drive.


## Abalation Studies

History-length helper and the Qwen ablation runner.

In [16]:
import time
from sklearn.metrics import f1_score


def build_history_query(t, step_idx, history_len):
    """history_len=None -> full history; otherwise last `history_len` actions."""
    history = t["action_reprs"][:step_idx]
    if history_len is not None:
        history = history[-history_len:]
    return f'task is: {t["confirmed_task"]}\nPrevious actions: {"; ".join(history)}'


def run_qwen_eval(history_len, label):
    """Mirrors Week 10's evaluation loop exactly, except the selector's prompt
    is rebuilt with `history_len` actions instead of the last-3 baked into
    ranker_test_examples. Shortlist selection (ranker scoring + shuffle) is
    untouched -- same seed, same order -- so results are directly comparable
    to Week 10's saved last-3 numbers."""
    eval_rng = random.Random(RANKER_SEED)
    results = []

    for inst in tqdm(scored_test_examples, desc=f"{label} eval"):
        t, action = test_action_lookup[(inst["annotation_id"], inst["action_uid"])]
        step_idx = next(i for i, a in enumerate(t["actions"]) if a["action_uid"] == action["action_uid"])

        gt_op = action["operation"]["op"]
        gt_value = action["operation"].get("value", "")
        gt_ids = {c["backend_node_id"] for c in action["pos_candidates"]}

        shortlist = list(rank_candidates(ranker_model, inst, RANKER_SHORTLIST_K))
        eval_rng.shuffle(shortlist)
        element_in_shortlist = any(cid in gt_ids for cid, _ in shortlist)

        letters = string.ascii_uppercase[: len(shortlist)]
        choices = [{"letter": l, "id": cid, "text": text} for l, (cid, text) in zip(letters, shortlist)]
        query = build_history_query(t, step_idx, history_len)
        prompt = build_prompt_from_shortlist(query, choices)

        t0 = time.perf_counter()
        try:
            raw_response = call_local_model(prompt)
            letter, pred_op, pred_value = parse_model_response(raw_response)
        except Exception as e:
            letter, pred_op, pred_value = "", "", ""
            raw_response = f"[ERROR: {e}]"
        latency_s = time.perf_counter() - t0

        chosen = next((c for c in choices if c["letter"] == letter), None)
        element_correct = bool(chosen) and chosen["id"] in gt_ids
        operation_correct = pred_op == gt_op
        value_correct = True if gt_op != "TYPE" else normalize_value(pred_value) == normalize_value(gt_value)
        step_success = element_correct and operation_correct and value_correct

        results.append({
            "annotation_id": inst["annotation_id"], "website": inst["website"],
            "step_idx": step_idx, "gt_op": gt_op, "pred_op": pred_op,
            "element_in_shortlist": element_in_shortlist, "element_correct": element_correct,
            "operation_correct": operation_correct, "value_correct": value_correct,
            "step_success": step_success, "latency_s": latency_s,
        })

    df = pd.DataFrame(results)
    df.to_csv(PROCESSED_DIR / f"week11_ablation_{label}_step_results.csv", index=False)
    shutil.copy(PROCESSED_DIR / f"week11_ablation_{label}_step_results.csv",
                BACKUP_DIR / f"week11_ablation_{label}_step_results.csv")

    task_success = df.groupby("annotation_id")["step_success"].all().mean()
    summary = {
        "label": label, "history_len": history_len,
        "n_steps": len(df), "n_tasks": df["annotation_id"].nunique(),
        "element_accuracy": df["element_correct"].mean(),
        "operation_f1_macro": f1_score(df["gt_op"], df["pred_op"], average="macro", zero_division=0),
        "step_success_rate": df["step_success"].mean(),
        "task_success_rate": task_success,
        "latency_mean_s": df["latency_s"].mean(),
        "latency_median_s": df["latency_s"].median(),
    }
    with open(PROCESSED_DIR / f"week11_ablation_{label}_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    shutil.copy(PROCESSED_DIR / f"week11_ablation_{label}_summary.json",
                BACKUP_DIR / f"week11_ablation_{label}_summary.json")

    print(f"=== {label} (history_len={history_len}): n={summary['n_tasks']} tasks / {summary['n_steps']} steps ===")
    for k, v in summary.items():
        if k not in ("label", "history_len", "n_steps", "n_tasks"):
            print(f"  {k:>22s}: {v:.4f}" if isinstance(v, float) else f"  {k:>22s}: {v}")
    return df, summary


Run the two new history-length conditions at full scale (~1036 steps each); runtime should be roughly the same order of magnitude as Week 10's own run.

In [17]:
ablation_last1_df, ablation_last1_summary = run_qwen_eval(history_len=1, label="last1")
ablation_full_df, ablation_full_summary = run_qwen_eval(history_len=None, label="fullhistory")


last1 eval:   0%|          | 0/1036 [00:00<?, ?it/s]

=== last1 (history_len=1): n=131 tasks / 1036 steps ===
        element_accuracy: 0.6274
      operation_f1_macro: 0.7757
       step_success_rate: 0.5975
       task_success_rate: 0.0840
          latency_mean_s: 1.4138
        latency_median_s: 1.3489


fullhistory eval:   0%|          | 0/1036 [00:00<?, ?it/s]

=== fullhistory (history_len=None): n=131 tasks / 1036 steps ===
        element_accuracy: 0.6641
      operation_f1_macro: 0.8144
       step_success_rate: 0.6332
       task_success_rate: 0.0916
          latency_mean_s: 1.4710
        latency_median_s: 1.4001


Pull in Week 10's already-saved last-3 result and print a clean 3-way comparison.

In [27]:
with open(PROCESSED_DIR / "week10_moonshot_summary.json") as f:
    week10_summary = json.load(f)

comparison = pd.DataFrame([
    {"history": "last-1", **{k: ablation_last1_summary[k] for k in
        ("element_accuracy", "operation_f1_macro", "step_success_rate", "task_success_rate")}},
    {"history": "last-3", **{k: week10_summary[k] for k in
        ("element_accuracy", "operation_f1_macro", "step_success_rate", "task_success_rate")}},
    {"history": "full-history", **{k: ablation_full_summary[k] for k in
        ("element_accuracy", "operation_f1_macro", "step_success_rate", "task_success_rate")}},
])
comparison


,history,element_accuracy,operation_f1_macro,step_success_rate,task_success_rate
0,last-1,0.627413,0.775662,0.597490,0.083969
1,last-3,0.651544,0.800704,0.622587,0.091603
2,full-history,0.664093,0.814384,0.633205,0.091603


Zero-shot latency benchmark, adding the gpt-4o-mini comparison point.

In [19]:
for _secret_name in ["OPENAI_API_KEY"]:
    if not os.environ.get(_secret_name):
        _val = get_secret(_secret_name)
        if _val:
            os.environ[_secret_name] = _val
            print(f"{_secret_name} loaded ({'*' * 6}{_val[-4:]})")

_ensure("openai")
from openai import OpenAI
_openai_client = OpenAI()


def call_model_openai(prompt, model="gpt-4o-mini"):
    resp = _openai_client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}], temperature=0,
    )
    return resp.choices[0].message.content


def run_zeroshot_latency_eval(history_len=3, label="zeroshot_gpt4o_mini", sample_size=None):
    eval_rng = random.Random(RANKER_SEED)
    steps = scored_test_examples if sample_size is None else \
        random.Random(RANKER_SEED).sample(scored_test_examples, sample_size)
    results = []

    for inst in tqdm(steps, desc=f"{label} eval"):
        t, action = test_action_lookup[(inst["annotation_id"], inst["action_uid"])]
        step_idx = next(i for i, a in enumerate(t["actions"]) if a["action_uid"] == action["action_uid"])

        gt_op = action["operation"]["op"]
        gt_value = action["operation"].get("value", "")
        gt_ids = {c["backend_node_id"] for c in action["pos_candidates"]}

        shortlist = list(rank_candidates(ranker_model, inst, RANKER_SHORTLIST_K))
        eval_rng.shuffle(shortlist)
        element_in_shortlist = any(cid in gt_ids for cid, _ in shortlist)

        letters = string.ascii_uppercase[: len(shortlist)]
        choices = [{"letter": l, "id": cid, "text": text} for l, (cid, text) in zip(letters, shortlist)]
        query = build_history_query(t, step_idx, history_len)
        prompt = build_prompt_from_shortlist(query, choices)

        t0 = time.perf_counter()
        try:
            raw_response = call_model_openai(prompt)
            letter, pred_op, pred_value = parse_model_response(raw_response)
        except Exception as e:
            letter, pred_op, pred_value = "", "", ""
            raw_response = f"[ERROR: {e}]"
        latency_s = time.perf_counter() - t0

        chosen = next((c for c in choices if c["letter"] == letter), None)
        element_correct = bool(chosen) and chosen["id"] in gt_ids
        operation_correct = pred_op == gt_op
        value_correct = True if gt_op != "TYPE" else normalize_value(pred_value) == normalize_value(gt_value)
        step_success = element_correct and operation_correct and value_correct

        results.append({
            "annotation_id": inst["annotation_id"], "website": inst["website"],
            "gt_op": gt_op, "pred_op": pred_op, "element_in_shortlist": element_in_shortlist,
            "element_correct": element_correct, "operation_correct": operation_correct,
            "value_correct": value_correct, "step_success": step_success, "latency_s": latency_s,
        })

    df = pd.DataFrame(results)
    df.to_csv(PROCESSED_DIR / f"week11_{label}_step_results.csv", index=False)
    shutil.copy(PROCESSED_DIR / f"week11_{label}_step_results.csv", BACKUP_DIR / f"week11_{label}_step_results.csv")

    summary = {
        "label": label, "n_steps": len(df), "n_tasks": df["annotation_id"].nunique(),
        "latency_mean_s": df["latency_s"].mean(), "latency_median_s": df["latency_s"].median(),
        "latency_p95_s": df["latency_s"].quantile(0.95),
    }
    with open(PROCESSED_DIR / f"week11_{label}_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    shutil.copy(PROCESSED_DIR / f"week11_{label}_summary.json", BACKUP_DIR / f"week11_{label}_summary.json")

    print(f"=== {label}: n={summary['n_tasks']} tasks / {summary['n_steps']} steps ===")
    for k, v in summary.items():
        if k not in ("label", "n_steps", "n_tasks"):
            print(f"  {k:>18s}: {v:.4f}s" if isinstance(v, float) else f"  {k:>18s}: {v}")
    return df, summary


LATENCY_SAMPLE_SIZE = None  # None = full 131-task split. Set e.g. 100 to subsample if time runs short.
zeroshot_latency_df, zeroshot_latency_summary = run_zeroshot_latency_eval(history_len=3, sample_size=LATENCY_SAMPLE_SIZE)


OPENAI_API_KEY loaded (******XxIA)


zeroshot_gpt4o_mini eval:   0%|          | 0/1036 [00:00<?, ?it/s]

=== zeroshot_gpt4o_mini: n=131 tasks / 1036 steps ===
      latency_mean_s: 0.6858s
    latency_median_s: 0.6554s
       latency_p95_s: 0.9318s


Final latency comparison: Qwen2.5-7B (local, 4-bit) vs. gpt-4o-mini (API).

In [20]:
qwen_latency_mean = pd.concat([ablation_last1_df, ablation_full_df])["latency_s"].mean()
print(f"Qwen2.5-7B (T4, 4-bit) mean per-step latency: {qwen_latency_mean:.3f}s")
print(f"gpt-4o-mini (API) mean per-step latency:        {zeroshot_latency_summary['latency_mean_s']:.3f}s")

Qwen2.5-7B (T4, 4-bit) mean per-step latency: 1.442s
gpt-4o-mini (API) mean per-step latency:        0.686s


## Summary

Load the two step-result files and define shared helpers.

In [22]:
import pandas as pd

moonshot_df = pd.read_csv(PROCESSED_DIR / "week10_moonshot_step_results.csv")
fallback_df = pd.read_csv(PROCESSED_DIR / "week7_fallback_pipeline_step_results.csv")

FAILURE_ORDER = ["success", "ranker_ceiling_miss", "wrong_element_picked",
                 "right_element_wrong_operation", "right_element_right_op_wrong_value"]


def classify_failure(row):
    if row["step_success"]:
        return "success"
    if not row["element_in_shortlist"]:
        return "ranker_ceiling_miss"
    if not row["element_correct"]:
        return "wrong_element_picked"
    if not row["operation_correct"]:
        return "right_element_wrong_operation"
    return "right_element_right_op_wrong_value"


Failure-type taxonomy, side by side.

In [23]:
def failure_taxonomy_table(df, label):
    failure_type = df.apply(classify_failure, axis=1)
    counts = failure_type.value_counts().reindex(FAILURE_ORDER, fill_value=0)
    pct = (counts / len(df) * 100).round(1)
    return pd.DataFrame({f"{label}_n": counts, f"{label}_pct": pct})


taxonomy_comparison = pd.concat([
    failure_taxonomy_table(moonshot_df, "moonshot"),
    failure_taxonomy_table(fallback_df, "fallback"),
], axis=1)
taxonomy_comparison


,moonshot_n,moonshot_pct,fallback_n,fallback_pct
success,645,62.3,472,45.6
ranker_ceiling_miss,61,5.9,61,5.9
wrong_element_picked,300,29.0,451,43.5
right_element_wrong_operation,10,1.0,25,2.4
right_element_right_op_wrong_value,20,1.9,27,2.6


Step success rate by ordinal position within task.

In [24]:
def step_position_table(df, label):
    df = df.copy()
    df["pos"] = df.groupby("annotation_id").cumcount() + 1
    df["pos_bucket"] = df["pos"].where(df["pos"] <= 3, 4).map({1: "step 1", 2: "step 2", 3: "step 3", 4: "step 4+"})
    g = df.groupby("pos_bucket")["step_success"].agg(["mean", "count"])
    return (g["mean"] * 100).round(1).rename(f"{label}_success_pct"), g["count"].rename(f"{label}_n")


m_pct, m_n = step_position_table(moonshot_df, "moonshot")
f_pct, f_n = step_position_table(fallback_df, "fallback")
step_position_comparison = pd.concat([m_pct, m_n, f_pct, f_n], axis=1).reindex(["step 1", "step 2", "step 3", "step 4+"])
step_position_comparison


,moonshot_success_pct,moonshot_n,fallback_success_pct,fallback_n
pos_bucket,,,,
step 1,65.6,131,43.5,131
step 2,73.3,131,51.1,131
step 3,66.1,127,48.8,127
step 4+,58.6,647,44.2,647


Task length vs. task success, with the independence-prediction check.

In [ ]:
def task_length_table(df, label):
    task_len = df.groupby("annotation_id").size()
    task_success = df.groupby("annotation_id")["step_success"].all()
    overall_p = df["step_success"].mean()

    length_bucket = task_len.where(task_len <= 6, 7)
    t = pd.DataFrame({"length": length_bucket, "success": task_success})
    g = t.groupby("length")["success"].agg(["mean", "count"])
    g["actual_pct"] = (g["mean"] * 100).round(1)
    g["predicted_pct"] = [round((overall_p ** min(n, 7)) * 100, 1) for n in g.index]
    g.index = [f"{n}" if n < 7 else "7+" for n in g.index]
    return g[["count", "actual_pct", "predicted_pct"]].rename(columns={"count": "n_tasks"})


print(f"--- Moonshot (overall step success = {moonshot_df['step_success'].mean():.1%}) ---")
display(task_length_table(moonshot_df, "moonshot"))

print(f"\n--- Fallback (overall step success = {fallback_df['step_success'].mean():.1%}) ---")
display(task_length_table(fallback_df, "fallback"))


Average task length, successful vs. failed tasks.

In [25]:
def avg_length_by_outcome(df, label):
    task_len = df.groupby("annotation_id").size()
    task_success = df.groupby("annotation_id")["step_success"].all()
    return pd.Series({
        f"{label}_avg_len_successful_tasks": round(task_len[task_success].mean(), 2),
        f"{label}_avg_len_failed_tasks": round(task_len[~task_success].mean(), 2),
    })


pd.concat([avg_length_by_outcome(moonshot_df, "moonshot"), avg_length_by_outcome(fallback_df, "fallback")])


,0
moonshot_avg_len_successful_tasks,4.08
moonshot_avg_len_failed_tasks,8.29
fallback_avg_len_successful_tasks,3.70
fallback_avg_len_failed_tasks,8.26


Worst-performing websites per pipeline (optional).

In [26]:
def website_failure_table(df, min_steps=15, top_n=5):
    g = df.groupby("website")["step_success"].agg(["mean", "count"])
    g["fail_rate_pct"] = ((1 - g["mean"]) * 100).round(1)
    return g[g["count"] >= min_steps].sort_values("fail_rate_pct", ascending=False)[["fail_rate_pct", "count"]].rename(columns={"count": "n_steps"}).head(top_n)


print("Moonshot — worst websites:")
display(website_failure_table(moonshot_df))
print("\nFallback — worst websites:")
display(website_failure_table(fallback_df))


Moonshot — worst websites:


,fail_rate_pct,n_steps
website,,
rei,60.9,46
cvs,60.0,15
ikea,45.0,20
bestbuy,44.4,153
menards,43.5,23



Fallback — worst websites:


,fail_rate_pct,n_steps
website,,
shopping.google,69.6,92
cvs,66.7,15
menards,65.2,23
cabelas,64.3,28
newegg,63.3,79
